In [6]:
_predict_df = "select (Version.[Version Name]*Product.[Product].[208821]*Time.FiscalWeek*SalesAccount.[Account]*Location.[Location]*{Measure.[DPSellOutUnitsActuals],Measure.[Mean Pricing Save PCT],Measure.[Placement Count],Measure.[Promotion Count],Measure.[DPSellOutPrice]});"

from o9_common_utils.O9DataLake import O9DataLake, ResourceType, DataSource,PluginSetting

# register inputs
predict_df = O9DataLake.register("predict_df",data_source = DataSource.LS, entity_type = ResourceType.IBPL, query = _predict_df,plugin_setting = PluginSetting.Inputs, spark = spark)
#systemreport = O9DataLake.register("newsysreportq4",data_source = DataSource.LIVEFRAME,entity_type = ResourceType.LIVEFRAME,plugin_setting = PluginSetting.Inputs)


# register slice dimension
O9DataLake.register("Product.[Product]", data_source = DataSource.LS, entity_type = ResourceType.IBPL, plugin_setting = PluginSetting.SliceDimension, spark = spark)

# register outputs
O9DataLake.register("output1",data_source = DataSource.LS,entity_type = ResourceType.IBPL, plugin_setting = PluginSetting.Outputs, spark = spark)


# register script params
script_params = O9DataLake.register({"var1":"50","var2":"Test1","var3":"newparam","var4":"80"}, data_source = DataSource.LS, plugin_setting = PluginSetting.ScriptParam)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
O9DataLake.inputs

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'predict_df': {'name': 'predict_df', 'resource_type': <ResourceType.IBPL: 'ibpl_query'>, 'data_source': <DataSource.LS: 'liveserver'>, 'query': 'select (Version.[Version Name]*Product.[Product].[208821]*Time.FiscalWeek*SalesAccount.[Account]*Location.[Location]*{Measure.[DPSellOutUnitsActuals],Measure.[Mean Pricing Save PCT],Measure.[Placement Count],Measure.[Promotion Count],Measure.[DPSellOutPrice]});', 'std_count_limit': '200000', 'df': DataFrame[VersionVersionName: string, ProductProduct: string, TimeFiscalWeek: string, SalesAccountAccount: string, LocationLocation: string, DPSellOutUnitsActuals: float, MeanPricingSavePCT: float, PlacementCount: float, PromotionCount: float, DPSellOutPrice: float]}}

In [8]:
type(predict_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<class 'pyspark.sql.dataframe.DataFrame'>

In [9]:
predict_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+--------------+--------------+-------------------+----------------+---------------------+------------------+--------------+--------------+--------------+
|VersionVersionName|ProductProduct|TimeFiscalWeek|SalesAccountAccount|LocationLocation|DPSellOutUnitsActuals|MeanPricingSavePCT|PlacementCount|PromotionCount|DPSellOutPrice|
+------------------+--------------+--------------+-------------------+----------------+---------------------+------------------+--------------+--------------+--------------+
|CurrentWorkingView|        208821|      W49-2015|                ALL|             ALL|                  NaN|               NaN|           2.0|           NaN|           NaN|
|CurrentWorkingView|        208821|      W50-2015|                ALL|             ALL|                  NaN|               NaN|           2.0|           NaN|           NaN|
|CurrentWorkingView|        208821|      W51-2015|                ALL|             ALL|                  NaN|               NaN|  

In [10]:
# fetching inputs
predict_df = O9DataLake.get('predict_df')
#input_df = O9DataLake.get('input_df')
#liveinput = O9DataLake.get('WeeklySales')

# fetching script params
value1 = O9DataLake.get_script_param("var1")
value2 = O9DataLake.get_script_param("var2")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
#user script

# package imports
import logging
from sklearn import tree

# initialize output variables
output1 = None
output2 = None

# initialize logger
logger = logging.getLogger('o9_logger')

logger.debug(f'predict_df dataframe:  {predict_df.count()}')
#logger.debug(f'input_df dataframe:  {input_df.shape}')
#logger.debug(f'liveinput dataframe:  {liveinput.shape}')
logger.debug(f'script param var1 value: {value1}')
logger.debug(f'script param var2 value: {value2}')


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-06-04 16:19:30,015 - o9_logger - DEBUG - predict_df dataframe:  160
2025-06-04 16:19:30,015 - o9_logger - DEBUG - script param var1 value: 50
2025-06-04 16:19:30,015 - o9_logger - DEBUG - script param var2 value: Test1

In [12]:
#in case storage push remains in stuck state, ps install google-cloud-storage package using jhub terminal
# /opt/conda/envs/k8qa_sparkk8org1tenant1/bin/pip install google-cloud-storage

from o9cloudutils import cloud_storage_utils
from o9cloudutils import user_storage_path
import os
import pandas as pd
import shutil
#from google.cloud import storage

STORAGE_BUCKET = f"test_o9cloudutils_from_pod"   # create bucket name for the slice


USER_STORAGE_PATH = user_storage_path
LOCAL_STORAGE_PATH = os.path.join(USER_STORAGE_PATH, STORAGE_BUCKET)   # create local storage path
TEST_FOLDER_PATH = os.path.join(LOCAL_STORAGE_PATH, "test")   # create subfolder path
if not os.path.exists(TEST_FOLDER_PATH):   # create local folder if it doesn't exist
    os.makedirs(TEST_FOLDER_PATH)


df = pd.DataFrame({"a":[1,2,3,4,5]})
df.to_csv(os.path.join(TEST_FOLDER_PATH,"single_col.csv"))

#Storage push

print('********Before storage_push*******')
print(os.listdir(LOCAL_STORAGE_PATH))
value = cloud_storage_utils.storage_push(STORAGE_BUCKET, LOCAL_STORAGE_PATH, overwrite=True)

if value:
    print("storage_push successful")
else:
    print("storage_push failed")
shutil.rmtree(LOCAL_STORAGE_PATH)


#Storage pull

print('********Before storage_pull*******')

if not os.path.exists(LOCAL_STORAGE_PATH):   # create local folder if it doesn't exist
    os.makedirs(LOCAL_STORAGE_PATH)

print('Files/Folders inside LOCAL STORAGE PATH')
print(os.listdir(LOCAL_STORAGE_PATH))

value = cloud_storage_utils.storage_pull(STORAGE_BUCKET, LOCAL_STORAGE_PATH, overwrite=True)

if value:
    print("storage_pull successful")

else:
    print("storage_pull failed")

print('********After storage_pull*******')
print(os.listdir(LOCAL_STORAGE_PATH))

outdf = pd.read_csv(os.path.join(LOCAL_STORAGE_PATH, 'test', 'single_col.csv'))
print('Pulled dataframe')
print(outdf.head())
out_df = pd.DataFrame()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

settings: {}
is_cluster_mode False
GCS Credential local path : None
using /tmp/tmpoez5o56t/20250604_161938_2d698f0d-a77b-44d8-9083-ea7079b4fc51 as local directory and /tmp/tmpzjgtr0g9/20250604_161938_2d698f0d-a77b-44d8-9083-ea7079b4fc51 as hdfs directory
***** = None
tenant_id is not valid
StorageType.HDFS
tenant_id is not valid
StorageType.amazon
********Before storage_push*******
['test']
storage_push successful
********Before storage_pull*******
Files/Folders inside LOCAL STORAGE PATH
[]
storage_pull successful
********After storage_pull*******
['test_o9cloudutils_from_pod.zip', 'test']
Pulled dataframe
   Unnamed: 0  a
0           0  1
1           1  2
2           2  3
3           3  4
4           4  5

In [53]:
print('plugin completed!')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

plugin completed!